## LLM is stateless

## openAI API_KEY 세팅

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


#  Single-Turn Conversation 
The Illusion of "memory"


API key found and looks good so far!


In [10]:
from openai import OpenAI

openai = OpenAI()

In [11]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm JINO!"}
    ]

In [12]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Hello JINO! How can I assist you today?'

## ask a follow-up question

In [13]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [14]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

"I don't know your name since you haven't shared it with me. How can I assist you today?"

## every call to an LLM is completely STATELESS. 

It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques 
to give the impression that the LLM has a "memory".

In [15]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm JINO!"},
    {"role": "assistant", "content": "Hi JINO! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [16]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

'Your name is JINO! How can I help you further?'

# Multi-Turn Conversation

In [ ]:
import openai

client = openai.OpenAI()
messages = []

def chat(user_input: str) -> str:
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

def reset():
    messages.clear()
    print("대화가 초기화되었습니다.")

In [24]:
# 대화 시작
reply = chat("안녕하세요 내 이름은 진오입니다.")
print(f"GPT: {reply}")
print(messages)

GPT: 안녕하세요, 진오님! 다시 인사해 주셔서 감사합니다. 오늘 기분은 어떠신가요?
[{'role': 'user', 'content': '안녕하세요 내 이름은 진오입니다.'}, {'role': 'assistant', 'content': '안녕하세요, 진오님! 만나서 반갑습니다. 어떻게 도와드릴까요?'}, {'role': 'user', 'content': '안녕하세요 내 이름은 진오입니다.'}, {'role': 'assistant', 'content': '안녕하세요, 진오님! 다시 인사해 주셔서 감사합니다. 오늘 기분은 어떠신가요?'}]


In [25]:
# 대화 시작
reply = chat("내 이름이 머냐?")
print(f"GPT: {reply}")
print(messages)

GPT: 진오님, 당신의 이름은 진오입니다!
[{'role': 'user', 'content': '안녕하세요 내 이름은 진오입니다.'}, {'role': 'assistant', 'content': '안녕하세요, 진오님! 만나서 반갑습니다. 어떻게 도와드릴까요?'}, {'role': 'user', 'content': '안녕하세요 내 이름은 진오입니다.'}, {'role': 'assistant', 'content': '안녕하세요, 진오님! 다시 인사해 주셔서 감사합니다. 오늘 기분은 어떠신가요?'}, {'role': 'user', 'content': '내 이름이 머냐?'}, {'role': 'assistant', 'content': '진오님, 당신의 이름은 진오입니다!'}]


# Manually manage conversation state



https://developers.openai.com/api/docs/guides/conversation-state#manually-manage-conversation-state


In [4]:

from openai import OpenAI

client = OpenAI()

history = [
    {
        "role": "user",
        "content": "농담 하나 해줘"
    }
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=history,
    store=False
)


print(response.output_text)
print(history)

물고기가 학교에 가면 어떤 과목을 배울까요?

"물리학!" 😄
[{'role': 'user', 'content': '농담 하나 해줘'}]


In [5]:
# Add the response to the conversation
history += [{"role": el.role, "content": el.content} for el in response.output]

history.append({ "role": "user", "content": "하나 더 해줘" })

second_response = client.responses.create(
    model="gpt-4o-mini",
    input=history,
    store=False
)

print(second_response.output_text)
print(history)

왜 자전거는 혼자서 못 가는 걸까요?

"두 바퀴가 필요하니까!" 😄
[{'role': 'user', 'content': '농담 하나 해줘'}, {'role': 'assistant', 'content': [ResponseOutputText(annotations=[], text='물고기가 학교에 가면 어떤 과목을 배울까요?\n\n"물리학!" 😄', type='output_text', logprobs=[])]}, {'role': 'user', 'content': '하나 더 해줘'}]
